# 07 — Korea DART Total-Accrual Replication (single cell, final)

Run the one code cell below **top to bottom**. It is self-contained and resumable.

**Before running:**
1. `pip install pandas numpy statsmodels finance-datareader`
2. Paste your free OpenDART key into `DART_KEY` (discard it after use).
3. DART blocks many overseas/cloud IPs — run from an unblocked network.

**Resumability:** if it stops (network, kernel restart), just run the cell again.
It reloads checkpoints from `../data` (`kr_corp_map.csv`, `kr_edgar_raw.csv`,
`kr_monthly_prices.csv`, `kr_factors.csv`) and continues.

**Outputs:** `kr_accounting_panel.csv`, `kr_tacc_portfolio_returns.csv`, and
`../results/tables/kr_tacc_hedge_alphas.csv`, plus printed quintile returns and
hedge alphas (P5 = low accrual, matching the paper).


## Result obtained by the authors (run locally with a DART key)

This notebook requires a DART API key and an unblocked network, so it is **not
executed here**. When run locally over 2013–2020 it produced:

**Korea TACC quintile mean monthly returns (%)**

| P1 (high) | P2 | P3 | P4 | P5 (low) |
|---|---|---|---|---|
| 0.905 | 1.134 | 1.086 | 1.271 | 0.457 |

**Low-minus-high (P5−P1) hedge alphas**

| Model | Alpha (%/mo) | t |
|---|---|---|
| CAPM | −0.36 | −1.29 |
| FF3 | −0.37 | −1.28 |

Firms per annual sort: 1,245–1,420. The hedge is negative and insignificant —
the accrual anomaly does **not** reproduce in Korea (see the manuscript).


In [ ]:
# ============================================================================
# Korea DART -> Total-Accrual Replication  (SINGLE CELL, final)
# ----------------------------------------------------------------------------
# Builds the Korean firm-level TACC/GPOA panel from OpenDART, downloads KRX
# monthly prices, pulls emerging-market factors, and reports TACC quintile
# returns + CAPM/FF3 hedge alphas. Mirrors the US pipeline exactly.
#
# HOW TO RUN
#   1) pip install pandas numpy statsmodels finance-datareader
#   2) paste your free OpenDART key into DART_KEY below (discard it afterwards)
#   3) run this one cell top to bottom. It is resumable: if it stops, just run
#      it again and it continues from the checkpoints in ../data.
#   NOTE: DART blocks many overseas/cloud IPs. Run from an unblocked network.
# ============================================================================
import urllib.request, urllib.parse, io, zipfile, json, time, os
import xml.etree.ElementTree as ET
import pandas as pd, numpy as np
import statsmodels.api as sm

# ---- CONFIG ---------------------------------------------------------------
DART_KEY = "YOUR_OPENDART_KEY_HERE"          # <-- paste key, discard after use
YEARS    = list(range(2013, 2021))
REPRT    = "11011"                            # annual report
FS_DIV   = "CFS"                              # consolidated; auto-fallback OFS
DATA_DIR = "../data"
TAB_DIR  = "../results/tables"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)
HDR = {"User-Agent": "Mozilla/5.0"}

def zfill6(s): return pd.Series(s).astype(str).str.replace(r"\.0$","",regex=True).str.zfill(6)
def zfill8(s): return pd.Series(s).astype(str).str.replace(r"\.0$","",regex=True).str.zfill(8)

# ===========================================================================
# STEP 1 — corp_code <-> stock_code map (listed firms only)
# ===========================================================================
MAP_CSV = f"{DATA_DIR}/kr_corp_map.csv"
if os.path.exists(MAP_CSV):
    corp_map = pd.read_csv(MAP_CSV, dtype=str)
else:
    url = f"https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={DART_KEY}"
    raw = urllib.request.urlopen(urllib.request.Request(url, headers=HDR), timeout=60).read()
    z = zipfile.ZipFile(io.BytesIO(raw)); root = ET.fromstring(z.read(z.namelist()[0]))
    rows = []
    for c in root.iter("list"):
        stock = (c.findtext("stock_code") or "").strip()
        if stock:                              # keep only KRX-listed firms
            rows.append((c.findtext("corp_code"), stock, c.findtext("corp_name")))
    corp_map = pd.DataFrame(rows, columns=["corp_code","stock_code","corp_name"])
    corp_map.to_csv(MAP_CSV, index=False)
corp_map["corp_code"]  = zfill8(corp_map["corp_code"])
corp_map["stock_code"] = zfill6(corp_map["stock_code"])
corp_map = corp_map.drop_duplicates("corp_code")
print(f"[1] listed firms: {len(corp_map)}")

# ===========================================================================
# STEP 2 — full financial statements per firm-year  (NI, OCF, Assets, Rev, COGS)
# ===========================================================================
WANT = {
    "ni":   ({"ifrs-full_ProfitLoss","ifrs_ProfitLoss"},
             {"당기순이익","당기순이익(손실)","분기순이익","연결당기순이익"}),
    "ocf":  ({"ifrs-full_CashFlowsFromUsedInOperatingActivities",
              "ifrs_CashFlowsFromUsedInOperatingActivities"},
             {"영업활동현금흐름","영업활동으로인한현금흐름"}),
    "at":   ({"ifrs-full_Assets","ifrs_Assets"}, {"자산총계"}),
    "rev":  ({"ifrs-full_Revenue","ifrs_Revenue"}, {"수익(매출액)","매출액","영업수익"}),
    "cogs": ({"ifrs-full_CostOfSales","ifrs_CostOfSales"}, {"매출원가"}),
}
def _num(x):
    try: return float(str(x).replace(",",""))
    except: return np.nan
def fetch(corp, year, fs=FS_DIV):
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json?" + urllib.parse.urlencode(
        {"crtfc_key":DART_KEY,"corp_code":corp,"bsns_year":str(year),
         "reprt_code":REPRT,"fs_div":fs})
    try:
        d = json.loads(urllib.request.urlopen(urllib.request.Request(url,headers=HDR),timeout=30).read())
    except Exception:
        return None
    if d.get("status") != "000":
        return fetch(corp, year, "OFS") if fs=="CFS" else None
    cur, prev = {}, {}
    for row in d.get("list", []):
        aid = (row.get("account_id") or "").strip()
        anm = (row.get("account_nm") or "").replace(" ","")
        for k,(ids,names) in WANT.items():
            if k in cur: continue
            if aid in ids or anm in {n.replace(" ","") for n in names}:
                cur[k]  = _num(row.get("thstrm_amount"))
                prev[k] = _num(row.get("frmtrm_amount"))
    return cur, prev

RAW_CSV = f"{DATA_DIR}/kr_edgar_raw.csv"
records, done = [], set()
if os.path.exists(RAW_CSV):
    prevdf = pd.read_csv(RAW_CSV, dtype={"corp_code":str})
    prevdf["corp_code"] = zfill8(prevdf["corp_code"])
    records = prevdf.to_dict("records")
    done = {(r["corp_code"], int(r["fyear"])) for r in records}
    print(f"[2] resuming: {len(records)} firm-years already collected")

corps = corp_map["corp_code"].tolist()
for i, corp in enumerate(corps):
    for y in YEARS:
        if (corp, y) in done: continue
        r = fetch(corp, y)
        if r and r[0]:
            cur, prev = r
            rec = {"corp_code": corp, "fyear": y}
            rec.update({k: cur.get(k) for k in WANT})
            rec["at_prev"] = prev.get("at")     # prior-year assets (deflator)
            records.append(rec)
        time.sleep(0.06)
    if i % 100 == 0:
        pd.DataFrame(records).to_csv(RAW_CSV, index=False)
        print(f"    {i}/{len(corps)} firms, {len(records)} firm-years", flush=True)
raw = pd.DataFrame(records)
raw["corp_code"] = zfill8(raw["corp_code"])
raw = raw.drop_duplicates(["corp_code","fyear"])
raw.to_csv(RAW_CSV, index=False)
print(f"[2] raw panel: {raw.shape}")

# ===========================================================================
# STEP 3 — join stock_code ONCE, build TACC & GPOA
# ===========================================================================
panel = raw.merge(corp_map[["corp_code","stock_code"]], on="corp_code", how="left")
panel["stock_code"] = zfill6(panel["stock_code"])
panel["tacc"] = (panel["ni"] - panel["ocf"]) / panel["at_prev"]
panel["gpoa"] = (panel["rev"] - panel["cogs"]) / panel["at_prev"]
panel = panel.replace([np.inf,-np.inf], np.nan).drop_duplicates(["corp_code","fyear"])
panel.to_csv(f"{DATA_DIR}/kr_accounting_panel.csv", index=False)
print(f"[3] panel: {panel.shape} | TACC: {panel['tacc'].notna().sum()} | GPOA: {panel['gpoa'].notna().sum()}")
print(panel[["tacc","gpoa"]].describe(percentiles=[.25,.5,.75]).round(4).to_string())

# ===========================================================================
# STEP 4 — KRX monthly prices via FinanceDataReader
# ===========================================================================
PX_CSV = f"{DATA_DIR}/kr_monthly_prices.csv"
if os.path.exists(PX_CSV):
    kr_px = pd.read_csv(PX_CSV, index_col=0, parse_dates=True)
    kr_px.columns = [str(c).zfill(6) for c in kr_px.columns]
    print(f"[4] prices loaded from cache: {kr_px.shape}")
else:
    import FinanceDataReader as fdr
    codes = sorted(panel.dropna(subset=["tacc"])["stock_code"].unique())
    closes = {}
    for i, sc in enumerate(codes):
        try:
            df = fdr.DataReader(sc, "2013-01-01", "2021-12-31")
            m = df["Close"].resample("ME").last().dropna()
            if len(m) > 24: closes[sc] = m
        except Exception:
            pass
        if i % 200 == 0: print(f"    prices {i}/{len(codes)} kept {len(closes)}", flush=True)
    kr_px = pd.DataFrame(closes).sort_index()
    kr_px.columns = [str(c).zfill(6) for c in kr_px.columns]
    kr_px.to_csv(PX_CSV)
    print(f"[4] prices: {kr_px.shape}")

# ===========================================================================
# STEP 5 — emerging-market factors (Kenneth French)
# ===========================================================================
FAC_CSV = f"{DATA_DIR}/kr_factors.csv"
if os.path.exists(FAC_CSV):
    fac = pd.read_csv(FAC_CSV, index_col=0, parse_dates=True)
else:
    url = ("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
           "Emerging_5_Factors_CSV.zip")
    raw_f = urllib.request.urlopen(urllib.request.Request(url, headers=HDR), timeout=30).read()
    z = zipfile.ZipFile(io.BytesIO(raw_f)); txt = z.read(z.namelist()[0]).decode("latin-1")
    rows = []
    for line in txt.splitlines():
        p = [x.strip() for x in line.split(",")]
        if len(p) >= 4 and p[0].isdigit() and len(p[0]) == 6:
            rows.append(p[:4])
    fac = pd.DataFrame(rows, columns=["ym","MktRF","SMB","HML"]).set_index("ym").astype(float)
    fac.index = pd.to_datetime(fac.index, format="%Y%m").to_period("M").to_timestamp()
    fac = fac/100.0
    fac.to_csv(FAC_CSV)
fac.index = fac.index.to_period("M").to_timestamp()
print(f"[5] factors: {fac.shape}")

# ===========================================================================
# STEP 6 — TACC quintiles + hedge alphas  (P5 = LOW accrual, paper convention)
# ===========================================================================
def winz(s, p=0.01): return s.clip(s.quantile(p), s.quantile(1-p))
acc = pd.read_csv(f"{DATA_DIR}/kr_accounting_panel.csv", dtype={"stock_code":str})
acc["stock_code"] = zfill6(acc["stock_code"])
acc["tacc_w"] = winz(acc["tacc"])

rets = kr_px.pct_change(fill_method=None).iloc[1:]
rets.index = rets.index.to_period("M").to_timestamp()
rets = rets.clip(-0.9, 4.0)
price_codes = set(rets.columns)

recs, counts = [], []
for fy in sorted(acc["fyear"].dropna().unique()):
    sub = acc[acc["fyear"]==fy].dropna(subset=["tacc_w"])
    sub = sub[sub["stock_code"].isin(price_codes)]
    if len(sub) < 50: continue
    sub = sub.copy()
    sub["q"] = 5 - pd.qcut(sub["tacc_w"], 5, labels=False, duplicates="drop")  # P5=low TACC
    counts.append((int(fy), len(sub)))
    hold = pd.date_range(f"{int(fy)+1}-07-01", f"{int(fy)+2}-06-30", freq="MS")
    for q in range(1,6):
        tks = [t for t in sub[sub["q"]==q]["stock_code"] if t in price_codes]
        if not tks: continue
        pr = rets.reindex(hold)[tks].mean(axis=1)
        for dt,v in pr.items():
            if pd.notna(v): recs.append((dt,q,v))

wide = pd.DataFrame(recs, columns=["date","q","ret"]).pivot_table(index="date",columns="q",values="ret")
wide.columns = [f"P{c}" for c in wide.columns]
wide["P5_P1"] = wide["P5"] - wide["P1"]
wide.to_csv(f"{DATA_DIR}/kr_tacc_portfolio_returns.csv")

d = wide.join(fac, how="inner").dropna(subset=["P5_P1"])
rows = []
for name, cols in {"CAPM":["MktRF"], "FF3":["MktRF","SMB","HML"]}.items():
    m = sm.OLS(d["P5_P1"], sm.add_constant(d[cols])).fit(cov_type="HAC", cov_kwds={"maxlags":3})
    rows.append({"model":name, "alpha_pct":round(m.params["const"]*100,3), "t":round(m.tvalues["const"],2)})
kr_alphas = pd.DataFrame(rows)
kr_alphas.to_csv(f"{TAB_DIR}/kr_tacc_hedge_alphas.csv", index=False)

print("\n[6] firms sorted per year:", counts)
print("    hedge months:", len(d))
print("\n=== Korea TACC quintile mean monthly returns (%) ===")
print((wide[[f"P{i}" for i in range(1,6)]].mean()*100).round(3).to_string())
print("\n=== Korea low-minus-high (P5-P1) hedge alphas ===")
print(kr_alphas.to_string(index=False))
